# NovaBank Digital Case Narrative

This case-narrative notebook teaches learners to read the digital-banking source packs, separate **Public Facts** from **Interpretation For Detection Patterns**, map evidence to a Detection pattern, and write a short investigation note against synthetic **NovaBank Digital** session and payment data.

Educational use only. This notebook supports fraud-detection curriculum design and is not legal, compliance, audit, investment, regulatory, or professional advice. Learner-facing views do not expose the protected answer keys.

## Setup

Load the digital-banking generator, learner-facing views, the session-context progressive view, and the Detection pattern registry. Synthetic data uses the `tiny` scale for fast execution.

In [ ]:
from banking_fraud_lab import (
    build_foundation_progressive_views,
    build_learner_facing_views,
    generate_digital_fraud_scenarios_world,
)
from banking_fraud_lab.progressive_views import NB_USER_SESSION_CONTEXT
from banking_fraud_lab.schema import PATTERN_IDS, PROTECTED_SCENARIO_ANSWER_KEYS

print("Frozen digital-banking Detection patterns present:", sorted(p for p in PATTERN_IDS if p.startswith(("digital_", "new_", "session_"))))

## Generate Learner-Facing Data

The canonical generator retains protected scenario answer keys for internal validation. The learner-facing tables loaded below exclude that protected table, and the session-context progressive view surfaces session telemetry for the investigation note.

In [ ]:
tables = generate_digital_fraud_scenarios_world(
    seed=42,
    scale="tiny",
    scenario_prevalence=0.5,
    noisy_outcome_rate=0.3,
)
learner_tables = build_learner_facing_views(tables)
progressive_views = build_foundation_progressive_views(learner_tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables, "Learner-facing views must exclude protected answer keys"
assert NB_USER_SESSION_CONTEXT.name in progressive_views, "Session-context view must be available"
session_context = progressive_views[NB_USER_SESSION_CONTEXT.name]
print("Sessions loaded:", len(session_context))
session_context.head()

## Step 1 — Read The Source Packs

Read the digital-banking source packs that anchor this case narrative:

- `docs/cases/source_packs/digital-scam-to-mule.md` supports the `digital_scam_to_mule` Detection pattern (authorised scam payment to a mule or fraudster account).
- `docs/cases/source_packs/digital-online-bank-control-failures.md` supports the `session_payment_velocity` Detection pattern (account-takeover with elevated session payment velocity).

These packs separate **Public Facts** (about the public source) from **Interpretation For Detection Patterns** (original learner-facing analysis). The case-narrative exercises below ask you to keep that separation in your own reasoning.

## Step 2 — Separate Facts From Interpretation

Using the source packs above, list two or three **facts** you can cite about the public source family (who published it, what it covers at a high level). Then list two or three **interpretations** — analytic claims about how a Detection pattern shows up in observable session or payment signals. Keep facts and interpretation in separate lists so a reviewer can see where evidence ends and analysis begins.

The relevant Detection patterns here are `digital_scam_to_mule`, `new_beneficiary_payment`, and `session_payment_velocity`; all are members of the frozen `PATTERN_IDS` registry.

In [ ]:
digital_patterns = sorted(
    p for p in PATTERN_IDS
    if p.startswith(("digital_", "new_", "session_"))
)
assert {'digital_scam_to_mule', 'new_beneficiary_payment', 'session_payment_velocity'} <= set(digital_patterns)
digital_patterns

## Step 3 — Map Evidence To A Detection Pattern

Examine the session-context view for account-control-failure signals. The view exposes raw session telemetry such as `is_vpn_or_proxy`, `asn_risk_score`, `auth_method`, and `channel`. The session-velocity and beneficiary-novelty features named in the source packs — such as `db_session_payment_count`, `db_is_new_beneficiary`, `db_is_shared_device`, and `db_is_early_life_account` — are emitted by the digital feature module (`notebooks/05_digital_session_and_payment_fraud/novabank_feature_engineering.ipynb`). The investigation question here is whether elevated session risk and payment velocity look different when read together rather than in isolation.

In [ ]:
session_risk = session_context.head(10)[[
    col for col in session_context.columns
    if col in {'session_id', 'user_id', 'channel', 'auth_method', 'is_vpn_or_proxy', 'asn_risk_score'}
]]
session_risk

## Step 4 — Write An Investigation Note

Write a short investigation note (four to six sentences) that:

1. Names the Detection pattern (`session_payment_velocity`, `digital_scam_to_mule`, or `new_beneficiary_payment`) your evidence maps to.
2. Cites the candidate data signal from the synthetic NovaBank Digital data.
3. States one limitation (for example, legitimate shared household devices or first-time payments to a new country).
4. Ends with a follow-up question for stakeholder discussion.

Use the glossary exactly: **User** is the digital login identity; **Client** is the legal customer. Educational framing only — no compliance instruction, and no claim that the synthetic data reconstructs any public matter.

In [ ]:
investigation_note_template = (
    "Detection pattern: {pattern}.\n\n"
    "Candidate signal: a session shows elevated session payment velocity together with a network-risk "
    "signal (is_vpn_or_proxy or a high asn_risk_score), or an early-life account shows rapid pass-through. "
    "This is reviewable evidence that the session or account warrants review.\n\n"
    "Limitation: legitimate shared household devices or first-time payments to a new country can share "
    "this shape, so the signal is decision support, not a finding about the User or Client.\n\n"
    "Follow-up question: how does authentication method (auth_method) at the time of the payment change "
    "the review priority?"
)
print(investigation_note_template.format(pattern="session_payment_velocity"))

## Cleanup

This notebook used only learner-facing views and the session-context progressive view. No protected answer keys were loaded, and no real User, account, or transaction data was used. Re-run from the top to refresh the synthetic data.

In [ ]:
del tables, learner_tables, progressive_views, session_context
print("Case-narrative notebook complete. Learner-facing views only; protected answer keys excluded.")